CashJY 特征取数 SQL 生成器

本 Notebook 根据“特征重要性”工作表中的“变量名”和“table”字段，生成可审核、可复用的特征取数 SQL 模板。

默认行为：

- 读取全部模型行；如只需要一个模型，将 SELECTED_MODELS 改为 ["loancntall"] 等。
- 保留 Excel 中字段首次出现顺序。
- 按来源表自动分配 t1、t2... 别名。
- 多来源字段自动生成原名、_v1、_v2... 别名。
- SQL 头部和每个 left join 前都会备注每张表获取的字段数和总获取字段数。


## tl;dr

运行导出单元格后，会在输出目录生成 SQL、字段映射、告警和来源表统计四个文件。生成的 SQL 只负责取数模板，不会连接或执行数据仓库查询。


## Context & Methods

### Key Assumptions

- 原始 Excel 不会被修改。
- table 单元格中使用逗号、中文逗号、分号或换行拆分多个来源表。
- 多来源候选默认全部输出，并通过别名和告警保留人工核对空间。
- 主表名、基础字段、关联键和分区字段是占位配置，需要按实际数仓表核对。
- SQL 分区占位符保持为 ${pt_beg} 和 ${pt_end}，便于下游替换。


In [ ]:
from collections import Counter
from pathlib import Path
import re

import pandas as pd

# ====== 可修改参数 ======
INPUT_XLSX = Path(r"D:\vscode\3_cashjy_loan_cnt\5.回溯打分表.xlsx")
SHEET_NAME = "特征重要性"
SELECTED_MODELS = []  # [] 表示全部模型，例如 ["loancntall"]

FEATURE_SET_NAME = "ng_pdl1st_cashjy_train_pct95"
DATASET_NAME = "ng_pdl1st_cashjy_train"

BASE_TABLE = "sample_table"
BASE_ALIAS = "t"
BASE_SELECT_COLUMNS = ["businessid", "jy_date", "fpd10_fz_dd"]

BASE_ID_COLUMN = "cid"
FEATURE_ID_COLUMN = "cid"
BASE_PARTITION_COLUMN = "pt_sub1d"
FEATURE_PARTITION_COLUMN = "pt"
PT_BEG_PLACEHOLDER = "$" + "{pt_beg}"
PT_END_PLACEHOLDER = "$" + "{pt_end}"

# 同一 Excel 行包含多张来源表时，默认全部保留并生成告警。
# 如人工确认某变量只应取某张表，可在此覆盖。
SOURCE_TABLE_OVERRIDES = {
    # "bank_cnt_90d": ["pb_biz_credit.keep_jch_ng_cid_multi_loan_feature_final"],
}

if Path.cwd().name == "risk_analysis":
    OUTPUT_DIR = Path.cwd() / "generated_cashjy_sql"
else:
    OUTPUT_DIR = Path.cwd() / "risk_analysis" / "generated_cashjy_sql"

SQL_OUTPUT_PATH = OUTPUT_DIR / "generated_feature_sql.sql"
MAPPING_OUTPUT_PATH = OUTPUT_DIR / "generated_feature_mapping.csv"
WARNINGS_OUTPUT_PATH = OUTPUT_DIR / "generated_feature_warnings.csv"
SUMMARY_OUTPUT_PATH = OUTPUT_DIR / "generated_feature_source_summary.csv"

def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).replace("\ufeff", "").strip()

def split_source_tables(value):
    text = clean_text(value)
    if not text:
        return []
    return [item.strip() for item in re.split(r"[,，;；\n]+", text) if item.strip()]

print("参数已加载")
print(f"输入 Excel: {INPUT_XLSX}")
print(f"输出目录: {OUTPUT_DIR}")


## Data

读取工作表并校验必要列，然后根据可选的模型筛选条件构造字段来源明细。


In [ ]:
if not INPUT_XLSX.exists():
    raise FileNotFoundError(f"找不到输入文件: {INPUT_XLSX}")

raw_all = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME, dtype=object)
raw_all.columns = [clean_text(column) for column in raw_all.columns]

required_columns = {"变量名", "table"}
missing_columns = sorted(required_columns - set(raw_all.columns))
if missing_columns:
    raise ValueError(f"工作表缺少必要列: {missing_columns}")

raw = raw_all.copy()
if SELECTED_MODELS:
    if "model" not in raw.columns:
        raise ValueError("设置了 SELECTED_MODELS，但工作表不存在 model 列")
    selected_models = {clean_text(model) for model in SELECTED_MODELS}
    raw = raw[raw["model"].map(clean_text).isin(selected_models)].copy()

print(f"工作表: {SHEET_NAME}")
print(f"原始数据行数: {len(raw_all)}")
print(f"筛选后数据行数: {len(raw)}")
if "model" in raw.columns:
    print("模型分布:")
    print(raw["model"].map(clean_text).value_counts(dropna=False).to_string())


In [ ]:
warnings = []
records = []

for excel_index, row in raw.iterrows():
    excel_row = int(excel_index) + 2
    variable_name = clean_text(row.get("变量名"))
    source_cell = clean_text(row.get("table"))
    model_name = clean_text(row.get("model"))
    online_name = clean_text(row.get("线上是否已经有了"))

    if not variable_name:
        warnings.append({
            "warning_type": "missing_variable",
            "variable_name": "",
            "source_table": "",
            "excel_row": excel_row,
            "detail": "变量名为空，已跳过",
        })
        continue

    source_tables = split_source_tables(source_cell)
    if variable_name in SOURCE_TABLE_OVERRIDES:
        source_tables = [clean_text(item) for item in SOURCE_TABLE_OVERRIDES[variable_name] if clean_text(item)]
        source_cell = "OVERRIDE: " + ", ".join(source_tables)

    if not source_tables:
        warnings.append({
            "warning_type": "missing_source",
            "variable_name": variable_name,
            "source_table": "",
            "excel_row": excel_row,
            "detail": "没有可用来源表，已跳过",
        })
        continue

    if len(source_tables) > 1:
        warnings.append({
            "warning_type": "same_row_multiple_sources",
            "variable_name": variable_name,
            "source_table": ", ".join(source_tables),
            "excel_row": excel_row,
            "detail": "同一 Excel 行包含多个来源表，默认全部保留",
        })

    for source_table in source_tables:
        records.append({
            "excel_row": excel_row,
            "model": model_name,
            "variable_name": variable_name,
            "source_cell": source_cell,
            "source_table": source_table,
            "online_name": online_name,
            "中文名": clean_text(row.get("中文名")),
            "一级分类": clean_text(row.get("一级分类")),
            "二级分类": clean_text(row.get("二级分类")),
        })

mapping_df = pd.DataFrame(records)
if mapping_df.empty:
    raise ValueError("没有生成任何字段映射，请检查变量名、table 列和模型筛选条件")

duplicate_mask = mapping_df.duplicated(["variable_name", "source_table"], keep="first")
for _, duplicate_row in mapping_df.loc[duplicate_mask].iterrows():
    warnings.append({
        "warning_type": "duplicate_variable_source",
        "variable_name": duplicate_row["variable_name"],
        "source_table": duplicate_row["source_table"],
        "excel_row": duplicate_row["excel_row"],
        "detail": "同一变量和来源表重复出现，已保留首次出现",
    })
mapping_df = mapping_df.loc[~duplicate_mask].reset_index(drop=True)

source_order = list(dict.fromkeys(mapping_df["source_table"].tolist()))
source_aliases = {source_table: f"t{index}" for index, source_table in enumerate(source_order, start=1)}
mapping_df["source_alias"] = mapping_df["source_table"].map(source_aliases)

seen_occurrences = Counter()
used_output_names = set(BASE_SELECT_COLUMNS)
output_names = []

for variable_name in mapping_df["variable_name"]:
    occurrence = seen_occurrences[variable_name]
    candidate = variable_name if occurrence == 0 else f"{variable_name}_v{occurrence}"

    while candidate in used_output_names:
        occurrence += 1
        candidate = f"{variable_name}_v{occurrence}"
        warnings.append({
            "warning_type": "output_name_collision",
            "variable_name": variable_name,
            "source_table": "",
            "excel_row": "",
            "detail": f"输出字段名冲突，已改为 {candidate}",
        })

    output_names.append(candidate)
    used_output_names.add(candidate)
    seen_occurrences[variable_name] = occurrence + 1

mapping_df["output_name"] = output_names

for variable_name, group in mapping_df.groupby("variable_name", sort=False):
    source_tables = list(dict.fromkeys(group["source_table"].tolist()))
    if len(source_tables) > 1:
        warnings.append({
            "warning_type": "same_variable_multiple_sources",
            "variable_name": variable_name,
            "source_table": ", ".join(source_tables),
            "excel_row": ", ".join(str(row) for row in group["excel_row"].tolist()),
            "detail": "同一变量来自多张来源表，已按出现顺序生成版本别名，请人工核对口径",
        })

warnings_df = pd.DataFrame(
    warnings,
    columns=["warning_type", "variable_name", "source_table", "excel_row", "detail"],
)

print(f"可输出特征字段数: {len(mapping_df)}")
print(f"唯一变量数: {mapping_df['variable_name'].nunique()}")
print(f"来源表数: {len(source_order)}")
print(f"告警数: {len(warnings_df)}")


In [ ]:
source_summary = (
    mapping_df.groupby(["source_alias", "source_table"], sort=False)
    .agg(
        feature_field_count=("output_name", "size"),
        variables=("output_name", lambda values: ", ".join(values)),
    )
    .reset_index()
)
source_summary["feature_field_count"] = source_summary["feature_field_count"].astype(int)

print("来源表字段统计:")
print(source_summary[["source_alias", "source_table", "feature_field_count"]].to_string(index=False))
print(f"\n总共获取的特征字段数: {int(source_summary['feature_field_count'].sum())}")


## Results

下面生成 SQL，并在 SQL 头部和每个来源表的 join 前写入字段统计备注。


In [ ]:
def render_sql():
    total_feature_fields = len(mapping_df)
    unique_variable_count = mapping_df["variable_name"].nunique()
    multi_source_variable_count = int(
        (mapping_df.groupby("variable_name")["source_table"].nunique() > 1).sum()
    )
    warning_count = len(warnings_df)

    header_lines = [
        "-- " + "=" * 60,
        "-- 取数 SQL 模板 (由 cashjy_feature_sql_generator.ipynb 生成)",
        f"-- 特征集: {FEATURE_SET_NAME}",
        f"-- 数据集: {DATASET_NAME}",
        f"-- 主表: {BASE_TABLE} (请按实际主表核对)  别名: {BASE_ALIAS}",
        (
            f"-- 关联键: {BASE_ALIAS}.{BASE_PARTITION_COLUMN} = tN.{FEATURE_PARTITION_COLUMN} "
            f"AND {BASE_ALIAS}.{BASE_ID_COLUMN} = tN.{FEATURE_ID_COLUMN}"
        ),
        f"-- 分区占位: {PT_BEG_PLACEHOLDER} ~ {PT_END_PLACEHOLDER}",
        f"-- 唯一变量数: {unique_variable_count} 个",
        f"-- 总共获取的特征字段数: {total_feature_fields} 个 (含多来源变量版本列，不含主表基础列)",
        f"-- 涉及源表: {len(source_summary)} 张",
        f"-- 告警数: {warning_count} 条，请结合 generated_feature_warnings.csv 人工核对",
        "-- 每个来源表获取的字段数统计:",
    ]

    for summary_row in source_summary.itertuples(index=False):
        header_lines.append(
            f"-- {summary_row.source_alias}: {summary_row.source_table} "
            f"-> 获取 {summary_row.feature_field_count} 个特征字段"
        )

    if warning_count:
        warning_counts = warnings_df["warning_type"].value_counts().to_dict()
        header_lines.append("-- 告警分类: " + ", ".join(
            f"{warning_type}={count}" for warning_type, count in warning_counts.items()
        ))
        for warning in warnings_df.itertuples(index=False):
            if warning.warning_type in {"same_variable_multiple_sources", "same_row_multiple_sources", "missing_source"}:
                header_lines.append(
                    f"-- [!] 变量 {warning.variable_name or '<空>'}: {warning.detail}; 来源: {warning.source_table or '<无>'}"
                )

    header_lines.append("-- " + "=" * 60)

    select_items = [f"{BASE_ALIAS}.{column}" for column in BASE_SELECT_COLUMNS]
    for row in mapping_df.itertuples(index=False):
        expression = f"{row.source_alias}.{row.variable_name}"
        if row.output_name != row.variable_name:
            expression += f" as {row.output_name}"
        select_items.append(expression)

    sql_lines = header_lines
    sql_lines.append("select")
    sql_lines.append("    " + ",\n    ".join(select_items))
    sql_lines.append(f"from {BASE_TABLE} as {BASE_ALIAS}")

    for source_table in source_order:
        source_alias = source_aliases[source_table]
        source_rows = mapping_df.loc[mapping_df["source_table"] == source_table]
        source_columns = ["cid", "pt"] + source_rows["variable_name"].tolist()
        source_columns = list(dict.fromkeys(source_columns))
        field_count = len(source_rows)

        sql_lines.extend([
            "",
            f"-- {source_alias}: {source_table}，获取特征字段数: {field_count}",
            "left join (",
            "    select " + ", ".join(source_columns),
            f"    from {source_table}",
            (
                f"    where {FEATURE_PARTITION_COLUMN} >= '{PT_BEG_PLACEHOLDER}' "
                f"and {FEATURE_PARTITION_COLUMN} <= '{PT_END_PLACEHOLDER}'"
            ),
            f") as {source_alias} on "
            f"{BASE_ALIAS}.{BASE_PARTITION_COLUMN} = {source_alias}.{FEATURE_PARTITION_COLUMN} "
            f"and {BASE_ALIAS}.{BASE_ID_COLUMN} = {source_alias}.{FEATURE_ID_COLUMN}",
        ])

    sql_lines.append(";")
    return "\n".join(sql_lines) + "\n"

generated_sql = render_sql()
print(generated_sql[:4000])
print("... SQL 预览已截断 ...")


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mapping_export_columns = [
    "excel_row", "model", "variable_name", "output_name",
    "source_alias", "source_table", "source_cell", "online_name",
    "中文名", "一级分类", "二级分类",
]
mapping_df[mapping_export_columns].to_csv(
    MAPPING_OUTPUT_PATH, index=False, encoding="utf-8-sig"
)
warnings_df.to_csv(WARNINGS_OUTPUT_PATH, index=False, encoding="utf-8-sig")
source_summary.to_csv(SUMMARY_OUTPUT_PATH, index=False, encoding="utf-8-sig")
SQL_OUTPUT_PATH.write_text(generated_sql, encoding="utf-8-sig")

print("已生成文件:")
for output_path in [
    SQL_OUTPUT_PATH,
    MAPPING_OUTPUT_PATH,
    WARNINGS_OUTPUT_PATH,
    SUMMARY_OUTPUT_PATH,
]:
    print(f"- {output_path} ({output_path.stat().st_size:,} bytes)")


In [ ]:
assert generated_sql.startswith("-- " + "=" * 60)
assert "select" in generated_sql.lower()
assert "left join (" in generated_sql.lower()
assert PT_BEG_PLACEHOLDER in generated_sql
assert PT_END_PLACEHOLDER in generated_sql
assert generated_sql.count("left join (") == len(source_summary)
assert len(mapping_df) == int(source_summary["feature_field_count"].sum())
assert SQL_OUTPUT_PATH.exists()
assert MAPPING_OUTPUT_PATH.exists()
assert WARNINGS_OUTPUT_PATH.exists()
assert SUMMARY_OUTPUT_PATH.exists()

sql_text = SQL_OUTPUT_PATH.read_text(encoding="utf-8-sig")
assert "-- 总共获取的特征字段数:" in sql_text
assert "-- 每个来源表获取的字段数统计:" in sql_text

print("校验通过")
print(f"SQL join 数: {generated_sql.count('left join (')}")
print(f"SQL 特征字段总数: {len(mapping_df)}")
print(f"SQL 来源表数: {len(source_summary)}")


## Takeaways

- 生成的 SQL 是模板，执行前请核对 BASE_TABLE、基础字段、关联键、分区字段和多来源变量口径。
- 若某字段不应从所有候选表输出，请在 SOURCE_TABLE_OVERRIDES 中配置唯一来源后重新运行。
- 重点查看 generated_feature_source_summary.csv，其中记录每张来源表获取的字段数；SQL 头部也会保留同样的统计备注。
